# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, leveraging its Croissant schema. The goal is to provide step-by-step guidance for metadata extraction, record overview, tabular loading, and basic exploratory data analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets in the dataset
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record set(s) in the dataset.")
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(no name)')}")
    if 'field' in rs and isinstance(rs['field'], list):
        print(f"  Fields (@id):")
        for f in rs['field']:
            # Each field is a dict
            if isinstance(f, dict):
                print(f"    - {f.get('@id')}")
            else:
                print(f"    - {f}")
    elif 'field' in rs:
        f = rs['field']
        if isinstance(f, dict):
            print(f"  Fields (@id):\n    - {f.get('@id')}")
        else:
            print(f"  Fields (@id):\n    - {f}")
    else:
        print("  No fields found in this record set.")
# For demonstration, pick the first record set's @id if available
if record_sets:
    first_record_set_id = record_sets[0]['@id']
    print("\nSampling first few records:")
    for i, rec in enumerate(dataset.records(record_set=first_record_set_id)):
        print(rec)
        if i > 2:
            break


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Define the record set @ids (as found above)
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    # Pull all records for each record set
    recs = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(recs)

# Show the columns in the first record set DataFrame as example
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Columns in record set {main_record_set_id}:")
    print(list(dataframes[main_record_set_id].columns))
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, attempt to select a representative numeric field by name
# Replace or customize if the actual field names differ.
import numpy as np

# Inspect a sample row
print("Sample rows from main record set:")
print(dataframes[main_record_set_id].head())

# Try to choose a numeric field automatically (e.g., 'age', 'interval_between_diagnoses_months', etc.)
numeric_candidates = [col for col in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][col].dtype in [np.float64, np.int64, float, int]]
if not numeric_candidates:
    # Try to suggest one by checking for numeric-looking names
    numeric_candidates = [col for col in dataframes[main_record_set_id].columns if 'age' in col.lower() or 'interval' in col.lower() or 'months' in col.lower()]

if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using '{numeric_field}' as numeric field.")
else:
    # Fall back to prompt user to fill in
    numeric_field = dataframes[main_record_set_id].columns[0]
    print(f"No obvious numeric field found. Using first column: {numeric_field}")

threshold = 60
try:
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field].astype(float) > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize this field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
    ) / (filtered_df[numeric_field].astype(float).std())
    print(f"Normalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
except Exception as e:
    print(f"Error filtering/norming numeric field: {e}")

# Group by a string/categorical field, e.g., 'sex', 'anatomical_site', etc.
possible_group_fields = [col for col in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][col].dtype == object]
group_field = None
for col in possible_group_fields:
    # Try to pick a field with >1 unique value and not 100% unique
    nunique = dataframes[main_record_set_id][col].nunique()
    if 1 < nunique < len(dataframes[main_record_set_id]):
        group_field = col
        break

if group_field and numeric_candidates:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nMean '{numeric_field}' by '{group_field}':")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Visualization 1: Histogram of the numeric field
plt.figure(figsize=(6, 4))
dataframes[main_record_set_id][numeric_field].astype(float).hist(bins=15)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Visualization 2: Bar plot of group vs mean(numeric field)
if group_field and numeric_candidates:
    plt.figure(figsize=(8, 4))
    plt.bar(grouped_df[group_field], grouped_df[numeric_field])
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:

- Load and access a Croissant-formatted FAIR² dataset with the `mlcroissant` library
- Explore available record sets and fields by their `@id`
- Extract tabular data using the dataset schema definitions
- Perform basic filtering, normalization, grouping, and plot visualizations

This workflow enables structured, reproducible exploration and processing of FAIR datasets for downstream ML and analysis tasks.